# 4.1 — Stat questions

Two probability problems. The written-out working is on paper (photo below); this notebook
does the same calculations two ways — **exactly**, and by **simulation** — so each answer
checks the other. If the exact number and the simulated number agree, the setup is probably
right.

The handwritten step-by-step working the question asks for:

<!-- Photo of the paper working. Drop it in assets/ and link it:
     ![Handwritten working for 4.1](assets/4.1-working.jpg) -->

## Question 1 — Chance guessing

A medium is given the car keys and wrist watches of **5 people** and tries to match each
watch to its key. If they are really just guessing, their answer is a random pairing — in
maths terms, a random permutation of 5 items. A "correct match" is a person whose watch and
key got paired correctly, which is called a **fixed point** of the permutation.

Two things are asked: the expected number of correct matches, and the probability of getting
at least one.

In [1]:
# Brute force: there are only 5! = 120 possible pairings, so just check them all.
from itertools import permutations
from fractions import Fraction

perms = list(permutations(range(5)))       # every way to match watches to keys
matches = [sum(1 for person, key in enumerate(p) if person == key) for p in perms]

expected = Fraction(sum(matches), len(perms))
at_least_one = Fraction(sum(1 for m in matches if m >= 1), len(perms))

print(f"total pairings:         {len(perms)}")
print(f"expected matches:       {expected}")
print(f"P(at least 1 match):    {at_least_one} = {float(at_least_one):.4f}")

total pairings:         120
expected matches:       1
P(at least 1 match):    19/30 = 0.6333


### Why those numbers

**Expected matches = 1.** Take person $i$ and let $X_i$ be 1 if their watch and key are
matched, 0 otherwise. Any given key is equally likely to land on any of the 5 people, so
$P(X_i = 1) = 1/5$. The total number of matches is $X = X_1 + \dots + X_5$, and expectation
adds up even when the events are not independent:

$$\mathbb{E}[X] = \sum_{i=1}^{5} \mathbb{E}[X_i] = 5 \times \tfrac{1}{5} = 1$$

Worth noticing: this gives 1 for *any* number of people. Five people or five hundred, a
guesser averages one correct match.

**P(at least 1) = 19/30.** Easier through the complement — count the pairings with *no*
correct match. Those are **derangements**, and for 5 items there are 44 of them:

$$P(\text{at least one}) = 1 - \frac{D_5}{5!} = 1 - \frac{44}{120} = \frac{76}{120} = \frac{19}{30} \approx 0.633$$

So a pure guesser gets at least one "hit" about **63%** of the time — which is the real point
of the question. A medium scoring one match has done nothing remarkable.

In [2]:
# Simulation check: shuffle the keys many times and count.
import numpy as np

rng = np.random.default_rng(0)
trials = 200_000
people = np.arange(5)

results = [np.sum(rng.permutation(people) == people) for _ in range(trials)]
results = np.array(results)

print(f"simulated mean matches:  {results.mean():.4f}   (exact 1)")
print(f"simulated P(>= 1 match): {(results >= 1).mean():.4f}   (exact {float(at_least_one):.4f})")

simulated mean matches:  1.0029   (exact 1)
simulated P(>= 1 match): 0.6336   (exact 0.6333)


## Question 2 — Bayesian calculation with two urns

Two urns, each holding 15 balls:

| Urn | Red | Green |
|---|---|---|
| A | 12 | 3 |
| B | 6 | 9 |

You are blindfolded and handed one of them, with no reason to think it is one rather than the
other — so the **prior** is $P(A) = P(B) = 1/2$. You draw a ball and it is red. That red ball
is evidence: urn A is redder, so drawing red should push your belief toward A. Bayes' rule is
how much.

In [3]:
# Set up the problem as fractions so the answers stay exact.
P_A = P_B = Fraction(1, 2)              # prior: the urn is equally likely to be either
red_given_A = Fraction(12, 15)          # urn A is 12/15 red
red_given_B = Fraction(6, 15)           # urn B is 6/15 red

# P(red) overall: the two ways a red ball could have happened, weighted by the prior.
P_red = red_given_A * P_A + red_given_B * P_B

# Bayes' rule: P(A|red) = P(red|A) P(A) / P(red)
P_A_given_red = red_given_A * P_A / P_red

print(f"P(red)      = {P_red}")
print(f"P(A | red)  = {P_A_given_red} = {float(P_A_given_red):.4f}")
print(f"P(B | red)  = {1 - P_A_given_red} = {float(1 - P_A_given_red):.4f}")

P(red)      = 3/5
P(A | red)  = 2/3 = 0.6667
P(B | red)  = 1/3 = 0.3333


### (i) Working

$$P(A \mid \text{red}) = \frac{P(\text{red} \mid A)\,P(A)}{P(\text{red})}
= \frac{\frac{12}{15} \cdot \frac{1}{2}}{\frac{12}{15}\cdot\frac{1}{2} + \frac{6}{15}\cdot\frac{1}{2}}
= \frac{0.4}{0.6} = \frac{2}{3}$$

The prior of $1/2$ moved to a posterior of $2/3$. One red ball is real but modest evidence —
it does not make you certain, because urn B contains red balls too.

### (ii) Drawing a second ball from the same urn

The second draw is not a fresh problem. The first red ball changed *which urn you think you
are holding*, and it also removed a red ball from that urn. Both effects matter:

$$P(\text{red}_2 \mid \text{red}_1) = P(A\mid\text{red})\cdot\tfrac{11}{14} + P(B\mid\text{red})\cdot\tfrac{5}{14}$$

using the updated $2/3$ and $1/3$ beliefs, and 14 balls left in whichever urn it is.

In [4]:
# Second draw, without replacement (the first red ball is now out of the urn).
P_B_given_red = 1 - P_A_given_red

second_red = (P_A_given_red * Fraction(11, 14)      # if it is urn A: 11 red of 14 left
              + P_B_given_red * Fraction(5, 14))    # if it is urn B: 5 red of 14 left

# If the ball were put back instead, the counts would not change:
second_red_replaced = P_A_given_red * red_given_A + P_B_given_red * red_given_B

print(f"P(2nd red | 1st red), without replacement = {second_red} = {float(second_red):.4f}")
print(f"P(2nd red | 1st red), with replacement    = {second_red_replaced} = {float(second_red_replaced):.4f}")

P(2nd red | 1st red), without replacement = 9/14 = 0.6429
P(2nd red | 1st red), with replacement    = 2/3 = 0.6667


Both readings are given because the question does not say whether the first
ball goes back. Taking it as a real draw (the ball stays out) gives **9/14 ≈ 0.643**; if it
were replaced, **2/3 ≈ 0.667**.

Either way the answer is **above** the unconditional chance of drawing red, which was
$P(\text{red}) = 0.6$. That is the whole point of the update: having seen one red ball, you
now believe you are more likely holding the red-heavy urn, so red is more likely next time
too.

In [5]:
# Simulation check: play the whole game many times and count.
trials = 200_000
first_red = 0
both_red = 0

for _ in range(trials):
    urn = ["R"] * 12 + ["G"] * 3 if rng.random() < 0.5 else ["R"] * 6 + ["G"] * 9
    draw = rng.permutation(urn)          # shuffle, then take the first two balls
    if draw[0] == "R":
        first_red += 1
        if draw[1] == "R":
            both_red += 1

print(f"kept {first_red:,} trials where the first ball was red")
print(f"simulated P(2nd red | 1st red) = {both_red / first_red:.4f}   (exact {float(second_red):.4f})")

kept 120,097 trials where the first ball was red
simulated P(2nd red | 1st red) = 0.6419   (exact 0.6429)


In [6]:
# And a direct check of P(A | red): among the times we drew red, how often was it urn A?
red_and_A = 0
red_total = 0

for _ in range(trials):
    is_A = rng.random() < 0.5
    urn = ["R"] * 12 + ["G"] * 3 if is_A else ["R"] * 6 + ["G"] * 9
    if rng.permutation(urn)[0] == "R":
        red_total += 1
        red_and_A += is_A

print(f"simulated P(A | red) = {red_and_A / red_total:.4f}   (exact {float(P_A_given_red):.4f})")

simulated P(A | red) = 0.6683   (exact 0.6667)


## Answers

| Question | Answer |
|---|---|
| 1. Expected number of correct matches | **1** |
| 1. P(at least one correct match) | **19/30 ≈ 0.633** |
| 2i. P(urn A given a red ball) | **2/3 ≈ 0.667** |
| 2ii. P(second red given first red) | **9/14 ≈ 0.643** without replacement (2/3 with) |

Every one was computed exactly and then reproduced by simulation, and the two agree to
roughly three decimal places in each case.